# Lab: Classical RAG: The Canonical Pipeline

## Setup

### Install Dependencies

In [1]:
!pip install pymupdf sentence-transformers faiss-cpu openai python-dotenv numpy

zsh:1: command not found: pip


### Import Libraries

In [2]:
import os
import warnings
import logging

import numpy as np
import pymupdf  # PyMuPDF — reads the PDF
import faiss  # the vector index
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from openai import OpenAI

### Silence Noisy Logs

In [3]:
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

### Load API Key & Build the Embedding Model

In [4]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

print("Key loaded.")

# The OpenAI client works with any OpenAI-compatible endpoint — here, OpenRouter.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

# The embedding model runs locally and needs no API key.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model ready:", embedder.get_sentence_embedding_dimension(), "dimensions")

Key loaded.


Embedding model ready: 384 dimensions


---
## Step 1 — Load the PDF

In [5]:
PDF_PATH = "data/sample_text_document.pdf"

document = pymupdf.open(PDF_PATH)
text = "\n".join(page.get_text() for page in document)

print(f"Loaded {document.page_count} page(s), {len(text):,} characters.")

Loaded 2 page(s), 4,065 characters.


---
## Step 2 — Chunk the Text

In [6]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size].strip())
        start += size - overlap
    return [chunk for chunk in chunks if chunk]

chunks = chunk_text(text)

print(f"Created {len(chunks)} chunks ({CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap).")
print(f"\nFirst chunk:\n{chunks[0][:200]}")

Created 10 chunks (500 chars, 50 overlap).

First chunk:
A Short Guide to Coastal Wetland Restoration
Synthetic Sample Document — Original Content, No Copyright Restrictions
1. Introduction
Coastal wetlands sit at the boundary between land and sea, absorbin


---
## Step 3 — Embed the Chunks

In [7]:
embeddings = embedder.encode(chunks, normalize_embeddings=True).astype("float32")

print("Embedding matrix shape:", embeddings.shape)

Embedding matrix shape: (10, 384)


---
## Step 4 — Build the FAISS Index

In [8]:
index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product on normalized vectors == cosine similarity
index.add(embeddings)

print("Vectors in index:", index.ntotal)

Vectors in index: 10


---
## Step 5 — Retrieve the Top-k Chunks

In [9]:
def retrieve(question, top_k=3):
    query_vector = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vector, top_k)
    return [(int(i), float(s)) for i, s in zip(indices[0], scores[0])]

QUESTION = "Why do restoration teams reintroduce tidal flow gradually instead of all at once?"

for rank, (i, score) in enumerate(retrieve(QUESTION), 1):
    print(f"\n--- Chunk {rank} (cosine {score:.3f}) ---")
    print(chunks[i][:200] + "...")


--- Chunk 1 (cosine 0.644) ---
tures is often
the single most effective step in a restoration project, since it allows the natural rise and fall of the tide to
resume moving sediment, nutrients, and organisms in and out of the site...

--- Chunk 2 (cosine 0.494) ---
rences determine whether a marsh plant community can establish
itself. Soil cores reveal whether the ground still holds buried peat or shell layers from a wetland that
existed before it was drained, w...

--- Chunk 3 (cosine 0.474) ---
soil. Phased approaches might open a small channel first, monitor erosion and sediment
deposition for a season, and then widen the opening in stages. This measured pace gives pioneer plant
species tim...


---
## Step 6 — Build the Prompt and Generate the Answer

In [10]:
def answer_question(question, top_k=3):
    hits = retrieve(question, top_k)
    context = "\n\n".join(f"[{n}] {chunks[i]}" for n, (i, _) in enumerate(hits, 1))
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip(), hits

answer, hits = answer_question(QUESTION)

print(f"Query: {QUESTION}")
print(f"\nAnswer: {answer}")

Query: Why do restoration teams reintroduce tidal flow gradually instead of all at once?

Answer: Restoration teams reintroduce tidal flow gradually because a sudden full breach can erode unconsolidated fill material before vegetation has a chance to stabilize the soil. Phased approaches allow teams to monitor erosion and sediment deposition, and give pioneer plant species time to colonize newly wetted areas as they emerge.


---
## Step 7 — Sweep the Two Knobs

In [11]:
for k in (1, 3, 5):
    _, hits_k = answer_question(QUESTION, top_k=k)
    context_chars = sum(len(chunks[i]) for i, _ in hits_k)
    print(f"top_k={k}: {len(hits_k)} chunks, {context_chars} context chars, top score {hits_k[0][1]:.3f}")

print()
for size in (100, 500, 2000):
    print(f"chunk_size={size}: {len(chunk_text(text, size=size, overlap=0))} chunks")

top_k=1: 1 chunks, 500 context chars, top score 0.644


top_k=3: 3 chunks, 1499 context chars, top score 0.644


top_k=5: 5 chunks, 2499 context chars, top score 0.644

chunk_size=100: 41 chunks
chunk_size=500: 9 chunks
chunk_size=2000: 3 chunks
